# NASA CMAPSS: RUL Prediction and FOPID Optimization

This notebook implements a complete pipeline for Remaining Useful Life (RUL) prediction using the NASA CMAPSS datasets (FD001, FD002, FD003, FD004) and optimizes a Fractional Order PID (FOPID) controller.

## Objectives
1. **RUL Prediction**: Load datasets FD001-FD004, process them according to their specific conditions (Sea Level vs Six Conditions), train MLP models, and optimize architectures using PSO.
2. **FOPID Control**: Simulate a fractional-order PID controller and optimize its parameters using PSO.
3. **Result Analysis**: Generate figures and summary tables for performance metrics.

## Data Requirements
Ensure the `data/CMAPSSData` folder contains the necessary text files.

In [ ]:
!pip install numpy pandas scikit-learn matplotlib seaborn

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from typing import Tuple, List, Dict
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 12, 'figure.dpi': 300})

## 1. Helper Functions
Functions for loading data, computing RUL, and preparing sequences.

In [ ]:
def load_cmapss(fd_path: str) -> pd.DataFrame:
    col_names = [f"col_{i}" for i in range(1, 27)]
    df = pd.read_csv(fd_path, sep=r"\s+", header=None, names=col_names)
    return df

def compute_rul(train_df: pd.DataFrame, clip_max: int = 125) -> pd.Series:
    grouped = train_df.groupby('col_1')['col_2'].max()
    max_cycle = train_df['col_1'].map(grouped)
    rul = (max_cycle - train_df['col_2']).clip(upper=clip_max)
    return rul

def build_sequences(df: pd.DataFrame, rul: pd.Series, seq_len: int = 30) -> Tuple[np.ndarray, np.ndarray]:
    feature_cols = df.columns[2:]
    units = df['col_1'].unique()
    sequences: List[np.ndarray] = []
    targets: List[float] = []
    for unit in units:
        unit_df = df[df['col_1'] == unit]
        unit_rul = rul[unit_df.index]
        scaler = MinMaxScaler()
        scaled = scaler.fit_transform(unit_df[feature_cols])
        for i in range(len(unit_df) - seq_len + 1):
            seq_x = scaled[i:i + seq_len].reshape(-1)
            seq_y = unit_rul.iloc[i + seq_len - 1]
            sequences.append(seq_x)
            targets.append(seq_y)
    X = np.array(sequences)
    y = np.array(targets)
    return X, y

## 2. PSO Implementation (MLP)
Particle Swarm Optimization logic for finding optimal neural network architectures.

In [ ]:
class Particle:
    def __init__(self, dim: int, bounds: Tuple[int, int]):
        self.position = np.random.uniform(bounds[0], bounds[1], size=dim)
        self.velocity = np.zeros(dim)
        self.best_position = self.position.copy()
        self.best_score = np.inf

def pso_optimize(X_train, X_val, y_train, y_val, n_particles=5, n_iter=5, bounds=(10, 100)):
    dim = 2
    particles = [Particle(dim, bounds) for _ in range(n_particles)]
    global_best_position = None
    global_best_score = np.inf
    score_history = []
    w, c1, c2 = 0.5, 1.5, 1.5

    for _ in range(n_iter):
        for p in particles:
            hidden_sizes = tuple(int(max(bounds[0], min(bounds[1], round(val)))) for val in p.position)
            model = MLPRegressor(hidden_layer_sizes=hidden_sizes, activation='relu', solver='adam', max_iter=200, random_state=42)
            model.fit(X_train, y_train)
            preds = model.predict(X_val)
            mse = mean_squared_error(y_val, preds)
            if mse < p.best_score:
                p.best_score = mse
                p.best_position = p.position.copy()
            if mse < global_best_score:
                global_best_score = mse
                global_best_position = p.position.copy()
            r1, r2 = np.random.rand(dim), np.random.rand(dim)
            p.velocity = w * p.velocity + c1 * r1 * (p.best_position - p.position) + c2 * r2 * (global_best_position - p.position)
            p.position += p.velocity
        score_history.append(global_best_score)
    best_hidden = tuple(int(round(max(bounds[0], min(bounds[1], val)))) for val in global_best_position)
    return best_hidden, global_best_score, score_history

def train_final_model(X_train, y_train, hidden_sizes):
    model = MLPRegressor(hidden_layer_sizes=hidden_sizes, activation='relu', solver='adam', max_iter=300, random_state=42)
    model.fit(X_train, y_train)
    return model

## 3. RUL Pipeline Execution
The function `process_dataset` handles loading, feature selection (dynamic based on dataset conditions), training, prediction, and plotting.

In [ ]:
def process_dataset(dataset_name: str) -> Dict:
    print(f"\n{'='*40}\nProcessing {dataset_name}\n{'='*40}")
    
    # Metadata & Settings check
    metadata = {
        'FD001': {'keep_settings': False},
        'FD002': {'keep_settings': True},
        'FD003': {'keep_settings': False},
        'FD004': {'keep_settings': True}
    }
    keep_settings = metadata.get(dataset_name, {}).get('keep_settings', False)
    
    train_path = f'data/CMAPSSData/train_{dataset_name}.txt'
    test_path = f'data/CMAPSSData/test_{dataset_name}.txt'
    try:
        train_df = load_cmapss(train_path)
        test_df = load_cmapss(test_path)
    except FileNotFoundError:
        print(f"Files for {dataset_name} not found.")
        return {}

    train_rul = compute_rul(train_df)
    
    # Feature Selection
    cols_to_drop = ['col_6', 'col_8', 'col_9', 'col_10', 'col_14', 'col_15', 'col_17', 'col_20', 'col_21', 'col_22', 'col_23']
    if not keep_settings:
        cols_to_drop.extend(['col_3', 'col_4', 'col_5'])
    
    train_df_red = train_df.drop(columns=cols_to_drop, errors='ignore')
    test_df_red = test_df.drop(columns=cols_to_drop, errors='ignore')
    
    X, y = build_sequences(train_df_red, train_rul, seq_len=30)
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Optimization & Training
    print("Optimizing MLP Architecture...")
    best_hidden, best_score, mlp_history = pso_optimize(X_train, X_val, y_train, y_val, n_particles=5, n_iter=5)
    print(f"Best Config: {best_hidden}, MSE: {best_score:.2f}")
    
    # Plot Convergence
    plt.figure(figsize=(8, 4))
    plt.plot(mlp_history, 'b-o')
    plt.title(f'PSO Convergence: {dataset_name}')
    plt.ylabel('MSE')
    plt.show()
    
    # Final Model
    model = train_final_model(X_train, y_train, best_hidden)
    preds = model.predict(X_val)
    
    mae = mean_absolute_error(y_val, preds)
    mse = mean_squared_error(y_val, preds)
    print(f"Final Validation MSE: {mse:.2f}")
    
    # Plot Predictions
    plt.figure(figsize=(10, 5))
    indices = np.argsort(y_val)
    plt.plot(y_val[indices], 'k-', label='True')
    plt.plot(preds[indices], 'r--', label='Pred')
    plt.title(f'RUL Prediction: {dataset_name}')
    plt.legend()
    plt.show()
    
    return {'Dataset': dataset_name, 'Best Hidden Layers': str(best_hidden), 'MSE': mse, 'MAE': mae}

# Executing for all datasets
results = []
for ds in ['FD001', 'FD002', 'FD003', 'FD004']:
    res = process_dataset(ds)
    if res:
        results.append(res)

results_df = pd.DataFrame(results)
print("\nSummary of RUL Prediction Results:")
display(results_df)

## 4. FOPID Controller Optimization
This section simulates a fractional-order PID controller connected to a second-order system and minimizes the Integral Square Error (ISE) using PSO.

In [ ]:
def simulate_fopid(ps_params: Tuple[float, float, float, float, float],
                   duration: float = 10.0, dt: float = 0.01) -> float:
    """Evalúa un controlador FOPID simplificado en un sistema de segundo orden."""
    Kp, Ki, Kd, lam, mu = ps_params
    x1 = 0.0  # posición del sistema
    x2 = 0.0  # velocidad del sistema
    integ = 0.0  # integral fraccionaria aproximada
    deriv_prev = 0.0
    ise = 0.0
    n_steps = int(duration / dt)
    for _ in range(n_steps):
        error = 1.0 - x1  # escalón unitario
        integ = integ + (error * dt) ** lam
        deriv = ((error - deriv_prev) / dt) ** mu
        deriv_prev = error
        u = Kp * error + Ki * integ + Kd * deriv
        # Modelo de sistema: x'' + x' + x = u
        a = u - x1 - x2
        x2 = x2 + a * dt
        x1 = x1 + x2 * dt
        ise += error ** 2 * dt
    return ise

def simulate_fopid_response(ps_params: Tuple[float, float, float, float, float],
                            duration: float = 10.0, dt: float = 0.01) -> Tuple[np.ndarray, np.ndarray]:
    """Simula la respuesta al escalón para graficar."""
    Kp, Ki, Kd, lam, mu = ps_params
    x1, x2 = 0.0, 0.0
    integ = 0.0
    deriv_prev = 0.0
    n_steps = int(duration / dt)
    t_values = np.linspace(0, duration, n_steps)
    y_values = []
    
    for _ in range(n_steps):
        error = 1.0 - x1
        integ = integ + (error * dt) ** lam
        deriv = ((error - deriv_prev) / dt) ** mu
        deriv_prev = error
        u = Kp * error + Ki * integ + Kd * deriv
        a = u - x1 - x2
        x2 = x2 + a * dt
        x1 = x1 + x2 * dt
        y_values.append(x1)
        
    return t_values, np.array(y_values)

def pso_fopid(n_particles: int = 5, n_iter: int = 5) -> Tuple[Tuple[float, float, float, float, float], float, List[float]]:
    """Optimiza los parámetros del FOPID mediante PSO minimizando el ISE."""
    dim = 5
    bounds = [(0.0, 2.0),  # Kp
              (0.0, 1.0),  # Ki
              (0.0, 1.0),  # Kd
              (0.5, 1.5),  # lambda (orden integral)
              (0.0, 1.0)]  # mu (orden derivativo)
    particles = []
    for _ in range(n_particles):
        pos = np.array([np.random.uniform(low, high) for low, high in bounds])
        p = {
            'position': pos,
            'velocity': np.zeros(dim),
            'best_position': pos.copy(),
            'best_cost': np.inf
        }
        particles.append(p)
        
    g_best_pos = None
    g_best_cost = np.inf
    cost_history = []
    w, c1, c2 = 0.5, 1.5, 1.5
    
    for iter_idx in range(n_iter):
        for p in particles:
            cost = simulate_fopid(tuple(p['position']))
            if cost < p['best_cost']:
                p['best_cost'] = cost
                p['best_position'] = p['position'].copy()
            if cost < g_best_cost:
                g_best_cost = cost
                g_best_pos = p['position'].copy()
        
        for p in particles:
            r1 = np.random.rand(dim)
            r2 = np.random.rand(dim)
            cognitive = c1 * r1 * (p['best_position'] - p['position'])
            social = c2 * r2 * (g_best_pos - p['position'])
            p['velocity'] = w * p['velocity'] + cognitive + social
            p['position'] += p['velocity']
            for i, (low, high) in enumerate(bounds):
                p['position'][i] = max(low, min(high, p['position'][i]))
                
        cost_history.append(g_best_cost)
        print(f"PSO FOPID Iter {iter_idx+1}/{n_iter}: Best ISE = {g_best_cost:.4f}")
        
    return tuple(g_best_pos), g_best_cost, cost_history

print("Starting PSO Optimization for FOPID...")
best_params, best_cost, fopid_history = pso_fopid(n_particles=15, n_iter=15)
print(f'Best FOPID params: {best_params}, ISE={best_cost:.4f}')

## 5. FOPID Results Analysis
Visualizing the optimization convergence and the step response comparison.

In [ ]:
# Plot FOPID Convergence
plt.figure(figsize=(8, 6))
plt.plot(range(1, len(fopid_history) + 1), fopid_history, 'g-o', linewidth=2)
plt.title('FOPID PSO Optimization Convergence', fontsize=14)
plt.xlabel('Iteration')
plt.ylabel('Best ISE Cost')
plt.grid(True)
plt.show()

# Plot FOPID Response Comparison
t_opt, y_opt = simulate_fopid_response(best_params)
mean_params = (1.0, 0.5, 0.5, 1.0, 0.5)
t_base, y_base = simulate_fopid_response(mean_params)

plt.figure(figsize=(10, 6))
plt.plot(t_opt, np.ones_like(t_opt), 'k--', label='Setpoint')
plt.plot(t_base, y_base, 'b:', label='Baseline Controller', linewidth=1.5)
plt.plot(t_opt, y_opt, 'r-', label='Optimized FOPID', linewidth=2)
plt.title('Step Response: Optimized vs Baseline', fontsize=14)
plt.xlabel('Time (s)')
plt.ylabel('System Output')
plt.legend()
plt.grid(True)
plt.show()